In [ ]:

import numpy as np
import torch
from torch import nn
import pathlib
import scipy
import os

try:
	import cvxpy as cp
except:
	print('No cvxpy package')


def lla2ecef(p, a = 6378137.0, e = 8.18191908426215e-2): # 0.0818191908426215, previous 8.1819190842622e-2
	p = p.copy().astype('float')
	p[:,0:2] = p[:,0:2]*np.array([np.pi/180.0, np.pi/180.0]).reshape(1,-1)
	N = a/np.sqrt(1 - (e**2)*np.sin(p[:,0])**2)
	# results:
	x = (N + p[:,2])*np.cos(p[:,0])*np.cos(p[:,1])
	y = (N + p[:,2])*np.cos(p[:,0])*np.sin(p[:,1])
	z = ((1-e**2)*N + p[:,2])*np.sin(p[:,0])
	return np.concatenate((x[:,None],y[:,None],z[:,None]), axis = 1)

def ecef2lla(x, a = 6378137.0, e = 8.18191908426215e-2):
	x = x.copy().astype('float')
	# https://www.mathworks.com/matlabcentral/fileexchange/7941-convert-cartesian-ecef-coordinates-to-lat-lon-alt
	b = np.sqrt((a**2)*(1 - e**2))
	ep = np.sqrt((a**2 - b**2)/(b**2))
	p = np.sqrt(x[:,0]**2 + x[:,1]**2)
	th = np.arctan2(a*x[:,2], b*p)
	lon = np.arctan2(x[:,1], x[:,0])
	lat = np.arctan2((x[:,2] + (ep**2)*b*(np.sin(th)**3)), (p - (e**2)*a*(np.cos(th)**3)))
	N = a/np.sqrt(1 - (e**2)*(np.sin(lat)**2))
	alt = p/np.cos(lat) - N
	# lon = np.mod(lon, 2.0*np.pi) # don't use!
	k = (np.abs(x[:,0]) < 1) & (np.abs(x[:,1]) < 1)
	alt[k] = np.abs(x[k,2]) - b
	return np.concatenate((180.0*lat[:,None]/np.pi, 180.0*lon[:,None]/np.pi, alt[:,None]), axis = 1)

def lla2ecef_diff(p, a = torch.Tensor([6378137.0]), e = torch.Tensor([8.18191908426215e-2]), device = 'cpu'):
	# x = x.astype('float')
	# https://www.mathworks.com/matlabcentral/fileexchange/7941-convert-cartesian-ecef-coordinates-to-lat-lon-alt
	a = a.to(device)
	e = e.to(device)
	# p = p.detach().clone().float().to(device) # why include detach here?
	pi = torch.Tensor([np.pi]).to(device)
	# p[:,0:2] = p[:,0:2]*torch.Tensor([pi/180.0, pi/180.0]).view(1,-1).to(device)
	N = a/torch.sqrt(1 - (e**2)*torch.sin(p[:,0]*pi/180.0)**2)
	# results:
	x = (N + p[:,2])*torch.cos(p[:,0]*pi/180.0)*torch.cos(p[:,1]*pi/180.0)
	y = (N + p[:,2])*torch.cos(p[:,0]*pi/180.0)*torch.sin(p[:,1]*pi/180.0)
	z = ((1-e**2)*N + p[:,2])*torch.sin(p[:,0]*pi/180.0)

	return torch.cat((x.view(-1,1), y.view(-1,1), z.view(-1,1)), dim = 1)

def ecef2lla_diff(x, a = torch.Tensor([6378137.0]), e = torch.Tensor([8.18191908426215e-2]), device = 'cpu'):
	# x = x.astype('float')
	# https://www.mathworks.com/matlabcentral/fileexchange/7941-convert-cartesian-ecef-coordinates-to-lat-lon-alt
	a = a.to(device)
	e = e.to(device)
	pi = torch.Tensor([np.pi]).to(device)
	b = torch.sqrt((a**2)*(1 - e**2))
	ep = torch.sqrt((a**2 - b**2)/(b**2))
	p = torch.sqrt(x[:,0]**2 + x[:,1]**2)
	th = torch.atan2(a*x[:,2], b*p)
	lon = torch.atan2(x[:,1], x[:,0])
	lat = torch.atan2((x[:,2] + (ep**2)*b*(torch.sin(th)**3)), (p - (e**2)*a*(torch.cos(th)**3)))
	N = a/torch.sqrt(1 - (e**2)*(torch.sin(lat)**2))
	alt = p/torch.cos(lat) - N
	# lon = np.mod(lon, 2.0*np.pi) # don't use!
	k = (torch.abs(x[:,0]) < 1) & (torch.abs(x[:,1]) < 1)
	alt[k] = torch.abs(x[k,2]) - b

	return torch.cat((180.0*lat[:,None]/pi, 180.0*lon[:,None]/pi, alt[:,None]), axis = 1)


def differential_evolution_location_trim(trv, locs_use, arv_p, ind_p, arv_s, ind_s, lat_range, lon_range, depth_range, time_range, sig_t = 1.5, weight = [1.0, 0.85], popsize = 75, maxiter = 1000, trim = 0.2, min_picks = 5, device = 'cpu', surface_profile = None, disp = True, vectorized = True):

	if (len(arv_p) + len(arv_s)) == 0:
		return np.nan*np.ones((1,3)), np.nan

	if surface_profile is None:

		def likelihood_estimate(x):

			# x = x.reshape(-1,3)
			if x.ndim == 1:
				x = x.reshape(-1,1)

			n_picks = len(ind_p) + len(ind_s)
			num_trim = int(np.floor(trim*n_picks))
			if n_picks - num_trim < min_picks: num_trim = n_picks - min_picks
			pred = trv(torch.Tensor(locs_use).to(device), torch.Tensor(x.T[:,0:3]).to(device)).cpu().detach().numpy() + x[3,:].reshape(-1,1,1)
			pred_vals = np.concatenate((pred[:,ind_p,0], pred[:,ind_s,1]), axis = 1) # , 1)
			logprob = ((trgt - pred_vals)**2)*weight_vec/(f_linear(pred_vals)**2) # , axis = 1)/n_picks
			irank = np.ones(logprob.shape)
			if (trim > 0)*(num_trim > 0)*(trim is not None)*(trim is not False):
				irank[np.arange(len(irank)).reshape(-1,1), np.argsort(logprob, axis = 1)[:,-num_trim::]] = 0.0
			logprob = -0.5*(irank*logprob).sum(1)/(n_picks - num_trim)

			return -1.0*logprob

	else:

		## Use surface to zero out probabilities in air
		x1_dim = np.unique(surface_profile[:,0])
		x2_dim = np.unique(surface_profile[:,1])
		nlen1, nlen2 = len(x1_dim), len(x2_dim)
		dx1, dx2 = np.diff(x1_dim)[0], np.diff(x2_dim)[0]

		def likelihood_estimate(x):

			# x = x.reshape(-1,3)
			if x.ndim == 1:
				x = x.reshape(-1,1)

			ind1 = np.maximum(np.minimum(np.floor((x[0,:] - x1_dim[0])/dx1).astype('int'), nlen1 - 1), 0)
			ind2 = np.maximum(np.minimum(np.floor((x[1,:] - x2_dim[0])/dx2).astype('int'), nlen2 - 1), 0)
			surf_elev = surface_profile[ind1 + ind2*nlen1, 2]
			prob_mask = np.ones(x.shape[1])
			prob_mask[x[2,:] > surf_elev] = 1e5


			n_picks = len(ind_p) + len(ind_s)
			num_trim = int(np.floor(0.2*n_picks))
			if n_picks - num_trim < min_picks: num_trim = n_picks - min_picks
			pred = trv(torch.Tensor(locs_use).to(device), torch.Tensor(x.T[:,0:3]).to(device)).cpu().detach().numpy() + x[3,:].reshape(-1,1,1)
			pred_vals = np.concatenate((pred[:,ind_p,0], pred[:,ind_s,1]), axis = 1) # , 1)
			logprob = ((trgt - pred_vals)**2)*weight_vec/(f_linear(pred_vals)**2) # , axis = 1)/n_picks
			irank = np.ones(logprob.shape)
			if (trim > 0)*(num_trim > 0)*(trim is not None)*(trim is not False):
				irank[np.arange(len(irank)).reshape(-1,1), np.argsort(logprob, axis = 1)[:,-num_trim::]] = 0.0
			logprob = -0.5*(irank*logprob).sum(1)/(n_picks - num_trim)


			return -1.0*(logprob*prob_mask)

	n_loc = locs_use.shape[1]
	n_picks = len(arv_p) + len(arv_s)
	trgt = np.concatenate((arv_p, arv_s), axis = 0).reshape(1, -1) # , 1)
	if len(weight) == n_picks:
		weight_vec = np.copy(np.array(weight)).reshape(1,-1)
	else:
		weight_vec = np.concatenate((weight[0]*np.ones(len(ind_p)), weight[1]*np.ones(len(ind_s))), axis = 0).reshape(1,-1)

	## Make sig_t adaptive to average distance of stations..
	f_linear = lambda t: sig_t*np.ones(t.shape)

	# pdb.set_trace()
	bounds = [(lat_range[0], lat_range[1]), (lon_range[0], lon_range[1]), (depth_range[0], depth_range[1]), (time_range[0], time_range[1])]
	optim = scipy.optimize.differential_evolution(likelihood_estimate, bounds, popsize = popsize, maxiter = maxiter, disp = disp, vectorized = vectorized)

	return optim.x[0:3].reshape(1,-1), optim.x[3].reshape(-1), likelihood_estimate(optim.x.reshape(-1,1))


def maximize_bipartite_assignment(cat, srcs, ftrns1, ftrns2, temporal_win = 10.0, spatial_win = 75e3, verbose = True):

	from scipy.spatial import cKDTree

	tree_t = cKDTree(srcs[:,3].reshape(-1,1))
	tree_s = cKDTree(ftrns1(srcs[:,0:3]))

	lp_t = tree_t.query_ball_point(cat[:,3].reshape(-1,1), r = temporal_win)
	lp_s = tree_s.query_ball_point(ftrns1(cat[:,0:3]), r = spatial_win)

	edges_cat_to_srcs = [np.array(list(set(lp_t[j]).intersection(lp_s[j]))) for j in range(cat.shape[0])]
	edges_cat_non_zero = np.where(np.array([len(edges_cat_to_srcs[j]) for j in range(cat.shape[0])]) > 0)[0]

	if sum([len(edges_cat_to_srcs[j]) for j in range(cat.shape[0])]) == 0:
		return np.array([]), [], [], [], []

	edges_cat_to_srcs = np.hstack([np.concatenate((j*np.ones(len(edges_cat_to_srcs[j])).reshape(1,-1), edges_cat_to_srcs[j].reshape(1,-1)), axis = 0) for j in edges_cat_non_zero]).astype('int')

	unique_cat_ind = np.sort(np.unique(edges_cat_to_srcs[0,:]))
	unique_src_ind = np.sort(np.unique(edges_cat_to_srcs[1,:]))

	nunique_cat = len(unique_cat_ind)
	nunique_src = len(unique_src_ind)

	temporal_diffs = cat[unique_cat_ind,3].reshape(-1,1) - srcs[unique_src_ind,3].reshape(1,-1)
	spatial_diffs = np.linalg.norm(ftrns1(cat[unique_cat_ind,0:3]).reshape(-1,1,3) - ftrns1(srcs[unique_src_ind,0:3]).reshape(1,-1,3), axis = 2)

	temporal_diffs[abs(temporal_diffs) > temporal_win] = np.inf
	spatial_diffs[abs(spatial_diffs) > spatial_win] = np.inf

	weights = np.exp(-0.5*(temporal_diffs**2)/(temporal_win**2))*np.exp(-0.5*(spatial_diffs**2)/(spatial_win**2))

	weights_unravel = weights.reshape(-1)
	weights_unravel[weights_unravel < 0.01] = -0.01 # This way, non-matches, are left unassigned

	## Make constraint matrix.
	# Each cat, assignment vector to at most one other sources.
	A1 = np.zeros((nunique_cat, len(weights_unravel)))
	for i in range(nunique_cat):
		A1[i,np.arange(nunique_src) + nunique_src*i] = 1.0

	## Make constraint matrix.
	# Each src, assignment vector to at most one other cat.
	A2 = np.zeros((nunique_src, len(weights_unravel)))
	for i in range(nunique_src):
		A2[i,np.arange(nunique_cat)*nunique_src + i] = 1.0

	A = np.concatenate((A1, A2), axis = 0)
	b = np.ones((A.shape[0],1))

	x = cp.Variable(A.shape[1], integer = True)
	prob = cp.Problem(cp.Minimize(-weights_unravel.T@x), constraints = [A@x <= b.reshape(-1), 0 <= x, x <= 1])
	prob.solve()
	assert prob.status == 'optimal', 'competitive assignment solution is not optimal'
	solution = np.round(x.value)

	assignment_vectors = solution.reshape(nunique_cat, nunique_src)
	assert(abs(assignment_vectors.sum(1) <= 1).min() == 1)
	assert(abs(assignment_vectors.sum(0) <= 1).min() == 1)

	results = []
	res = []
	for i in range(nunique_cat):
		i1 = np.where(assignment_vectors[i,:] > 0)[0]

		# print('temporal diff %0.4f'%temporal_diffs[i, i1[0]])
		# print('spatial diff %0.4f'%spatial_diffs[i, i1[0]])

		if len(i1) > 0:
			results.append(np.array([unique_cat_ind[i], unique_src_ind[i1[0]]]).reshape(1,-1))
			res.append((cat[unique_cat_ind[i],0:4] - srcs[unique_src_ind[i1[0]],0:4]).reshape(1,-1))

			if verbose == True:
				print('temporal diff %0.4f'%temporal_diffs[i, i1[0]])
				print('spatial diff %0.4f'%spatial_diffs[i, i1[0]])

	results = np.vstack(results)
	res = np.vstack(res)

	return results, res, assignment_vectors, unique_cat_ind, unique_src_ind

def load_travel_time_neural_network(path_to_file, ftrns1, ftrns2, n_ver_load, phase = 'p_s', device = 'cuda', method = 'relative pairs', corrs = None, locs_corr = None, return_model = False, use_physics_informed = False):

	if use_physics_informed == False:

		# from module import TravelTimes
		seperator = '\\' if '\\' in path_to_file else '/'

		# z = np.load(path_to_file + '1D_Velocity_Models_Regional' + seperator + 'travel_time_neural_network_%s_losses_ver_%d.npz'%(phase, n_ver_load))
		z = np.load(path_to_file + '1D_Velocity_Models_Regional' + seperator + 'travel_time_neural_network_%s_losses_ver_%d.npz'%(phase, n_ver_load))

		n_phases = len(z['v_mean'])
		scale_val = float(z['scale_val'])
		trav_val = float(z['trav_val'])
		z.close()

		m = TravelTimes(ftrns1, ftrns2, scale_val = scale_val, trav_val = trav_val, n_phases = n_phases, device = device).to(device)
		m.load_state_dict(torch.load(path_to_file + '/1D_Velocity_Models_Regional/travel_time_neural_network_%s_ver_%d.h5'%(phase, n_ver_load), map_location = torch.device(device)))

		if return_model == False:
			m.eval()

		if method == 'relative pairs':

			trv = lambda sta_pos, src_pos: m.forward_relative(sta_pos, src_pos, method = 'pairs')

		if method == 'direct':

			trv = lambda sta_pos, src_pos: m.forward_relative(sta_pos, src_pos, method = 'direct')

		if return_model == True:

			return m

		else:

			return trv

	else:

		# from module import VModel, TravelTimesPN
		seperator = '\\' if '\\' in path_to_file else '/'

		if os.path.isfile(path_to_file + '1D_Velocity_Models_Regional' + seperator + 'travel_time_neural_network_physics_informed_%s_losses_ver_%d.npz'%(phase, n_ver_load)) == True:
			z = np.load(path_to_file + '1D_Velocity_Models_Regional' + seperator + 'travel_time_neural_network_physics_informed_%s_losses_ver_%d.npz'%(phase, n_ver_load))
		else:
			z = np.load(path_to_file + 'travel_time_neural_network_physics_informed_%s_losses_ver_%d.npz'%(phase, n_ver_load))


		n_phases = len(z['v_mean'])
		v_mean, scale_params = z['v_mean'], z['scale_params']
		# scale_val = float(z['scale_val'])
		# trav_val = float(z['trav_val'])
		z.close()

		max_dist, max_time, vp_max, vs_min, scale_norm_factor, conversion_factor = scale_params

		norm_pos = lambda x: x/max_dist
		norm_time = lambda t: t/max_time
		norm_vel = lambda v: v/vp_max

		inorm_pos = lambda x: x*max_dist
		inorm_time = lambda t: t*max_time
		inorm_vel = lambda v: v*vp_max

		m = TravelTimesPN(ftrns1, ftrns2, n_phases = n_phases, v_mean = v_mean, norm_pos = norm_pos, inorm_pos = inorm_pos, inorm_time = inorm_time, norm_vel = norm_vel, conversion_factor = conversion_factor, corrs = corrs, locs_corr = locs_corr, device = device).to(device)
		if os.path.isfile(path_to_file + '/1D_Velocity_Models_Regional/travel_time_neural_network_physics_informed_%s_ver_%d.h5'%(phase, n_ver_load)) == True:
			m.load_state_dict(torch.load(path_to_file + '/1D_Velocity_Models_Regional/travel_time_neural_network_physics_informed_%s_ver_%d.h5'%(phase, n_ver_load), map_location = torch.device(device)))
		else:
			m.load_state_dict(torch.load(path_to_file + 'travel_time_neural_network_physics_informed_%s_ver_%d.h5'%(phase, n_ver_load), map_location = torch.device(device)))


		if return_model == False:
			m.eval()

		if method == 'relative pairs':

			trv = lambda sta_pos, src_pos: m(sta_pos, src_pos, method = 'pairs')

		if method == 'direct':

			trv = lambda sta_pos, src_pos: m(sta_pos, src_pos, method = 'direct')

		if return_model == True:

			return m

		else:

			return trv

class TravelTimesPN(nn.Module):

	def __init__(self, ftrns1, ftrns2, n_phases = 1, n_srcs = 0, n_hidden = 50, n_embed = 10, v_mean = np.array([6500.0, 3400.0]), norm_pos = None, inorm_pos = None, inorm_time = None, norm_vel = None, conversion_factor = None, corrs = None, locs_corr = None, device = 'cuda'):
		super(TravelTimesPN, self).__init__()

		## Relative offset prediction [2]
		self.fc1_1 = nn.Linear(3 + n_phases + n_embed, n_hidden)
		self.fc1_2 = nn.Linear(n_hidden, n_hidden)
		self.fc1_3 = nn.Linear(n_hidden, n_hidden)
		# self.fc1_4 = nn.Linear(n_hidden, n_phases)
		self.activate1_1 = lambda x: torch.sin(x)
		self.activate1_2 = lambda x: torch.sin(x)
		self.activate1_3 = lambda x: torch.sin(x)

		## Absolute position prediction [3]
		self.fc2_1 = nn.Linear(6 + n_phases + n_embed, n_hidden)
		self.fc2_2 = nn.Linear(n_hidden, n_hidden)
		self.fc2_3 = nn.Linear(n_hidden, n_hidden)
		# self.fc2_4 = nn.Linear(n_hidden, n_phases)
		self.activate2_1 = lambda x: torch.sin(x)
		self.activate2_2 = lambda x: torch.sin(x)
		self.activate2_3 = lambda x: torch.sin(x)

		self.merge = nn.Sequential(nn.Linear(2*n_hidden, n_hidden), nn.PReLU(), nn.Linear(n_hidden, n_phases))

		## Embed source [3]
		# self.fc3_1 = nn.Linear(3 + 2 + 1, n_hidden)
		self.fc3_1 = nn.Linear(3, n_hidden)
		self.fc3_2 = nn.Linear(n_hidden, n_hidden)
		self.fc3_3 = nn.Linear(n_hidden, n_hidden)
		self.fc3_4 = nn.Linear(n_hidden, n_embed)
		self.activate3_1 = lambda x: torch.sin(x)
		self.activate3_2 = lambda x: torch.sin(x)
		self.activate3_3 = lambda x: torch.sin(x)

		## Projection functions
		self.ftrns1 = ftrns1
		self.ftrns2 = ftrns2
		# self.scale = torch.Tensor([scale_val]).to(device) ## Might want to scale inputs before converting to Tensor
		# self.tscale = torch.Tensor([trav_val]).to(device)
		self.v_mean = torch.Tensor(v_mean).to(device)
		self.v_mean_norm = torch.Tensor(norm_vel(v_mean)).to(device)
		self.device = device
		self.norm_pos = norm_pos
		self.inorm_pos = inorm_pos
		self.inorm_time = inorm_time
		self.norm_vel = norm_vel
		self.conversion_factor = conversion_factor
		self.vmodel = VModel(n_phases = n_phases, n_embed = n_embed, device = device).to(device)
		self.mask = torch.Tensor([0.0, 0.0, 1.0]).reshape(1,-1).to(device)
		self.scale_angles = torch.Tensor([180.0, 180.0]).reshape(1,-1).to(device) ## Make these adaptive
		self.scale_depths = torch.Tensor([300e3]).reshape(1,-1).to(device)
		if locs_corr is not None:
			self.tree_corr = cKDTree(ftrns1(torch.Tensor(locs_corr).to(device)).cpu().detach().numpy())
			self.corrs = torch.Tensor(corrs).to(device)
			self.use_corr = True
		else:
			self.use_corr = False

		if n_srcs > 0:
			self.reloc_x = nn.Parameter(torch.zeros((n_srcs, 3))) # .to(device)
			self.reloc_t = nn.Parameter(torch.zeros((n_srcs, 1))) # .to(device)

		# self.Tp_average

	def fc1_block(self, x):

		x1 = self.activate1_1(self.fc1_1(x))
		x = self.activate1_2(self.fc1_2(x1)) + x1
		x1 = self.activate1_3(self.fc1_3(x)) + x

		return x1 # self.fc1_4(x1)

	def fc2_block(self, x):

		x1 = self.activate2_1(self.fc2_1(x))
		x = self.activate2_2(self.fc2_2(x1)) + x1
		x1 = self.activate2_3(self.fc2_3(x)) + x

		return x1 # self.fc2_4(x1)

	def fc3_block(self, x):

		x1 = self.activate3_1(self.fc3_1(x))
		x = self.activate3_2(self.fc3_2(x1)) + x1
		x1 = self.activate3_3(self.fc3_3(x)) + x

		return self.fc3_4(x1)

	def embed_src(self, src):

		return self.fc3_block(self.norm_pos(self.ftrns1(src)))

	def src_proj(self, src):

		return self.norm_pos(self.ftrns1(src))


	def forward(self, sta, src, method = 'pairs', train = False):

		# embed_src = self.fc3_block(self.norm_pos(self.ftrns1(src)))
		# embed_src = self.embed_src(src*self.mask)
		embed_src = self.embed_src(src)

		if method == 'direct':

			sta_proj = self.norm_pos(self.ftrns1(sta))
			src_proj = self.norm_pos(self.ftrns1(src))

			if train == True:
				src_proj = Variable(src_proj, requires_grad = True)

			base_val = self.conversion_factor*torch.norm(sta_proj - src_proj, dim = 1, keepdim = True)/self.v_mean_norm.reshape(1,-1)

			pred1 = self.fc1_block( torch.cat((sta_proj - src_proj, base_val, embed_src), dim = 1) )
			pred2 = self.fc2_block( torch.cat((sta_proj, src_proj, base_val, embed_src), dim = 1) )
			pred = self.merge(torch.cat((pred1, pred2), dim = 1))

			if train == True:

				return base_val, pred, src_proj, embed_src

			else:

				if self.use_corr == True:
					imatch = self.tree_corr.query(self.ftrns1(sta).cpu().detach().numpy())[1]
					return torch.relu(self.inorm_time(base_val + pred) + self.corrs[imatch,:])

				else:

					return torch.relu(self.inorm_time(base_val + pred))

				# return torch.relu(self.inorm_time(base_val + pred))

		elif method == 'pairs':

			## First, create all pairs of srcs and recievers

			src_repeat = self.norm_pos(self.ftrns1(src)).repeat_interleave(len(sta), dim = 0) # /self.scale
			sta_repeat = self.norm_pos(self.ftrns1(sta)).repeat(len(src), 1) # /self.scale
			src_embed_repeat = embed_src.repeat_interleave(len(sta), dim = 0)

			if train == True:
				src_repeat = Variable(src_repeat, requires_grad = True)

			base_val = self.conversion_factor*(torch.norm(sta_repeat - src_repeat, dim = 1, keepdim = True)/self.v_mean_norm.reshape(1,-1)) # .reshape(len(src), len(sta), -1)

			pred1 = self.fc1_block(torch.cat((sta_repeat - src_repeat, base_val, src_embed_repeat), dim = 1)) # .reshape(len(src), len(sta), -1)
			pred2 = self.fc2_block(torch.cat((sta_repeat, src_repeat, base_val, src_embed_repeat), dim = 1)) # .reshape(len(src), len(sta), -1)
			pred = self.merge(torch.cat((pred1, pred2), dim = 1)).reshape(len(src), len(sta), -1)

			if train == True:

				return base_val.reshape(len(src), len(sta), -1), pred, src_repeat.reshape(len(src), len(sta), -1), src_embed_repeat.reshape(len(src), len(sta), -1)

			else:

				if self.use_corr == True:
					imatch = self.tree_corr.query(self.ftrns1(sta).cpu().detach().numpy())[1]
					return torch.relu(self.inorm_time(base_val.reshape(len(src), len(sta), -1) + pred) + self.corrs[imatch,:].unsqueeze(0))

				return torch.relu(self.inorm_time(base_val.reshape(len(src), len(sta), -1) + pred))

				# return torch.relu(self.inorm_time(base_val.reshape(len(src), len(sta), -1) + pred))

class VModel(nn.Module):

	def __init__(self, n_phases = 2, n_hidden = 50, n_embed = 10, device = 'cuda'): # v_mean = np.array([6500.0, 3400.0]), norm_pos = None, inorm_pos = None, inorm_time = None, norm_vel = None, conversion_factor = None,
		super(VModel, self).__init__()

		## Relative offset prediction [2]
		self.fc1_1 = nn.Linear(3 + n_embed, n_hidden)
		self.fc1_2 = nn.Linear(n_hidden, n_hidden)
		self.fc1_3 = nn.Linear(n_hidden, n_hidden)
		self.fc1_4 = nn.ModuleList()
		for j in range(n_phases):
			self.fc1_4.append(nn.Linear(n_hidden, 1))
			# self.fc1_41 = nn.Linear(n_hidden, 1)
			# self.fc1_42 = nn.Linear(n_hidden, 1)
		self.activate1_1 = lambda x: torch.sin(x)
		self.activate1_2 = lambda x: torch.sin(x)
		self.activate1_3 = lambda x: torch.sin(x)
		self.activate = nn.Softplus()
		self.mask = torch.zeros((1, 3)).to(device) # + n_embed)).to(device)
		self.mask[0,2] = 1.0
		self.n_phases = n_phases

	def fc1_block(self, x):

		# x = x*torch.Tensor([0.0, 0.0, 1.0]).reshape(1,-1).to(x.device)
		x1 = self.activate1_1(self.fc1_1(x))
		x = self.activate1_2(self.fc1_2(x1)) + x1
		x1 = self.activate1_3(self.fc1_3(x)) + x
		# out = [self.activate(self.fc1_4[j](x1)) for j in range(self.n_phases)]

		return [self.activate(self.fc1_4[j](x1)) for j in range(self.n_phases)]

	def forward(self, src, embed):

		out = self.fc1_block(torch.cat((src, embed), dim = 1))
		lout = [out[0]]
		for j in range(1, self.n_phases):
			lout.append(out[0]*out[j])
		# out[:,1] = out[:,0]*out[:,1] ## Vs is a fraction of Vp

		return torch.cat(lout, dim = 1)


def apply_location(trv, stas_all, locs_all, sta_names, pick_times, phase_types, lat_range, lon_range, depth_range, surface_profile = None, device = 'cpu'): # max_relative_error: 0.2 ## 0.15 corresponds to 15% maximum relative error allowed # min_time_buffer: 1.25

	# use_quality_control = False, max_relative_error = 0.2, min_time_buffer = 1.25

	imatch_sta = np.array([np.where(stas_all == name)[0][0] for name in sta_names])
	i1, i2 = np.where(phase_types == 0)[0], np.where(phase_types == 1)[0]

	arv_p, arv_s = pick_times[i1], pick_times[i2]
	ind_p, ind_s = imatch_sta[i1], imatch_sta[i2]

	min_time = pick_times.min()
	time_spread = pick_times.max() - pick_times.min()

	ind_unique_arrivals = np.sort(np.unique(np.concatenate((ind_p, ind_s), axis = 0)).astype('int'))

	perm_vec_arrivals = -1*np.ones(locs_all.shape[0]).astype('int')
	perm_vec_arrivals[ind_unique_arrivals] = np.arange(len(ind_unique_arrivals))
	locs_use_slice = locs_all[ind_unique_arrivals]
	ind_p_perm_slice = perm_vec_arrivals[ind_p]
	ind_s_perm_slice = perm_vec_arrivals[ind_s]
	if len(ind_p_perm_slice) > 0:
		assert(ind_p_perm_slice.min() > -1)
	if len(ind_s_perm_slice) > 0:
		assert(ind_s_perm_slice.min() > -1)

	# pdb.set_trace()

	# xmle, origin, logprob = differential_evolution_location_trim(trv, locs_use_slice, arv_p - min_time, ind_p_perm_slice, arv_s - min_time, ind_s_perm_slice, lat_range_extend, lon_range_extend, depth_range, [-time_spread/2.0, time_spread/2.0], surface_profile = surface_profile, device = device)
	xmle, origin, logprob = differential_evolution_location_trim(trv, locs_use_slice, arv_p - min_time, ind_p_perm_slice, arv_s - min_time, ind_s_perm_slice, lat_range, lon_range, depth_range, [-time_spread, time_spread], surface_profile = surface_profile, device = device)

	origin = min_time + origin
	pred_out = trv(torch.Tensor(locs_use_slice).to(device), torch.Tensor(xmle).to(device)).cpu().detach().numpy() + origin

	res_p = pred_out[0,ind_p_perm_slice,0] - arv_p
	res_s = pred_out[0,ind_s_perm_slice,1] - arv_s

	rms_residual = np.concatenate((res_p, res_s), axis = 0)
	rms_residual = np.linalg.norm(rms_residual)/np.sqrt(len(rms_residual))

	xmle = np.concatenate((xmle, np.array([origin]).reshape(1,-1)), axis = 1)

	arv_p1 = np.concatenate((arv_p.reshape(-1,1), ind_p.reshape(-1,1), pred_out[:,ind_p_perm_slice,0].reshape(-1,1), res_p.reshape(-1,1)), axis = 1)
	arv_s1 = np.concatenate((arv_s.reshape(-1,1), ind_s.reshape(-1,1), pred_out[:,ind_s_perm_slice,0].reshape(-1,1), res_s.reshape(-1,1)), axis = 1)

	return xmle, rms_residual, pred_out, arv_p1, arv_s1

def download_catalog(lat_range, lon_range, min_magnitude, startime, endtime, t0 = None, client = 'NCEDC', include_arrivals = False):

	from obspy.core import UTCDateTime
	from obspy.clients.fdsn import Client

	if t0 is None:
		t0 = UTCDateTime(2000, 1, 1)

	client = Client(client)
	# pdb.set_trace()
	cat_l = client.get_events(starttime = startime, endtime = endtime, minlatitude = lat_range[0], maxlatitude = lat_range[1], minlongitude = lon_range[0], maxlongitude = lon_range[1], minmagnitude = min_magnitude, includearrivals = include_arrivals, orderby = 'time-asc')

	# t0 = UTCDateTime(2021,4,1) ## zero time, for relative processing.
	time = np.array([cat_l[i].origins[0].time - t0 for i in np.arange(len(cat_l))])
	latitude = np.array([cat_l[i].origins[0].latitude for i in np.arange(len(cat_l))])
	longitude = np.array([cat_l[i].origins[0].longitude for i in np.arange(len(cat_l))])
	depth = np.array([cat_l[i].origins[0].depth for i in np.arange(len(cat_l))])
	mag = np.array([cat_l[i].magnitudes[0].mag for i in np.arange(len(cat_l))])
	event_type = np.array([cat_l[i].event_type for i in np.arange(len(cat_l))])

	cat = np.hstack([latitude.reshape(-1,1), longitude.reshape(-1,1), -1.0*depth.reshape(-1,1), time.reshape(-1,1), mag.reshape(-1,1)])

	return cat, cat_l, event_type


In [ ]:
import numpy as np
from location_scripts import *


z = np.load('WestCoast_stations.npz')
locs, stas, rbest, mn = z['locs'], z['stas'], z['rbest'], z['mn']
z.close()

z = np.load('WestCoast_region.npz')
lat_range, lon_range, depth_range, deg_pad = z['lat_range'], z['lon_range'], z['depth_range'], z['deg_pad']
z.close()
lat_range_extend = [lat_range[0] - deg_pad, lat_range[0] + deg_pad]
lon_range_extend = [lon_range[0] - deg_pad, lon_range[0] + deg_pad]

## Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rbest_cuda = torch.Tensor(rbest).to(device)
mn_cuda = torch.Tensor(mn).to(device)

## Projection functions
earth_radius = 6378137.0
ftrns1 = lambda x: (rbest @ (lla2ecef(x) - mn).T).T # just subtract mean
ftrns2 = lambda x: ecef2lla((rbest.T @ x.T).T + mn) # just subtract mean

ftrns1_diff = lambda x: (rbest_cuda @ (lla2ecef_diff(x, device = device) - mn_cuda).T).T # just subtract mean
ftrns2_diff = lambda x: ecef2lla_diff((rbest_cuda.T @ x.T).T + mn_cuda, device = device) # just subtract mean


In [1]:
## Load travel time neural network
path_to_file = './'
n_ver_trv_time_model_load = 1
locs_corr, corrs = None, None
use_physics_informed = True
trv = load_travel_time_neural_network(path_to_file, ftrns1_diff, ftrns2_diff, n_ver_trv_time_model_load, locs_corr = locs_corr, corrs = corrs, use_physics_informed = use_physics_informed, device = device)
f_apply_location = lambda pick_times, sta_names, phase_types: apply_location(trv, stas, locs, sta_names, pick_times, phase_types, lat_range, lon_range, depth_range, surface_profile = None, device = device)

##### Note: #####
## Can locate earthquakes by creating the 3 numpy (1d) arrays of:
## [1]. pick_times (the absolute pick times of each station)
## [2]. sta_name (the station names of each pick (as Ntx.Sta code for network and station codes))
## [3]. phase_type (the vector of phase types of each pick, with 0 for P waves and 1 for S waves)
## Then call: "f_apply_location(pick_times, sta_names, phase_types)"
## Solution is stored in "xmle" (1x4 array): (lat, lon, depth, origin time); depth is in meters with negative below sea level


NameError: name 'load_travel_time_neural_network' is not defined

In [ ]:
##### Example 1: #######

# Set random sta_names, pick_times, phase_types per event.
# E.g, set a random source and stations, create synthetic arrival times, and locate

random_src = np.array([np.random.uniform(d[0], d[1]) for d in [lat_range, lon_range, depth_range, [-20, 20]]]).reshape(1,-1)
random_stations = np.random.choice(len(stas), size = 30, replace = False)

# Synthetic travel times (note: input to "trv" is stations x sources; output is sources x stations x phase_type (2))
trv_times = trv(torch.Tensor(locs[:,0:3]).to(device), torch.Tensor(random_src).to(device)).cpu().detach().numpy() + random_src[:,3].reshape(-1,1,1)

trv_times += np.random.normal(size = trv_times.shape)*(0.015*(trv_times - random_src[:,3].reshape(-1,1,1))) ## Random noise 1.5% scaled to travel times

pick_times_p = trv_times[0,random_stations,0] ## Extract pick times
pick_times_s = trv_times[0,random_stations,1]

pick_times = np.concatenate((pick_times_p, pick_times_s), axis = 0)
sta_names = np.concatenate((stas[random_stations], stas[random_stations]), axis = 0)
phase_types = np.concatenate((np.zeros(len(pick_times_p)), np.ones(len(pick_times_p))), axis = 0)

print('\nUsing picks (%d):\n'%len(pick_times))
print(pick_times)
print(sta_names)
print(phase_types)
print('\n')

xmle, rms_residual, pred_out, arv_p1, arv_s1 = f_apply_location(pick_times, sta_names, phase_types)
# xmle, rms_residual, pred_out, arv_p1, arv_s1 = apply_location(trv, stas, locs, sta_names, pick_times, phase_types, lat_range, lon_range, depth_range, surface_profile = None, device = device)

print('\nRandom source (true): %0.3f lat, %0.3f lon, %0.3f depth, %0.3f s'%(tuple([j for j in random_src[0,:]])))
print('Located source: %0.3f lat, %0.3f lon, %0.3f depth, %0.3f s'%(tuple([j for j in xmle[0,:]])))


In [ ]:
##### Example 2: #######

## Locate a real event previously detected by NCEDC

from obspy.core import UTCDateTime
# Download event from obspy
start_time = UTCDateTime(2008, 1, 15) # March 6, 2008
end_time = UTCDateTime(2008, 1, 18)
cat, cat_l, _ = download_catalog(lat_range, lon_range, 1.0, start_time, end_time, t0 = start_time, include_arrivals = True)
print('\nDownloaded %d events'%len(cat))

src_index = 0 ## Note: use depths as meters with negative below sea level (e.g., -30,000 is 30 km depth)
src_index = np.random.choice(len(cat))

ref_src = np.array([cat_l[src_index].origins[0].latitude, cat_l[src_index].origins[0].longitude, -1.0*cat_l[src_index].origins[0].depth]).reshape(1,-1)
origin_time = cat_l[src_index].origins[0].time
origin_time_seconds = origin_time - start_time
ref_src = np.concatenate((ref_src, np.array([origin_time_seconds]).reshape(1,1)), axis = 1)

## Extract catalog pick data
pick_times = np.array([cat_l[src_index].picks[j].time - start_time for j in range(len(cat_l[src_index].picks))])
sta_names = np.array([cat_l[src_index].picks[j].waveform_id.network_code + '.' + cat_l[src_index].picks[j].waveform_id.station_code for j in range(len(cat_l[src_index].picks))])
assert(min([cat_l[src_index].picks[j].phase_hint == 'P' or cat_l[src_index].picks[j].phase_hint == 'S' for j in range(len(cat_l[src_index].picks))]) == 1)
phase_types = np.array([(0 if cat_l[src_index].picks[j].phase_hint == 'P' else 1) for j in range(len(cat_l[src_index].picks))])
iuse_picks = np.where([s in stas for s in sta_names])[0]
print('Using %d of %d picks (matched to stations "stas")'%(len(iuse_picks), len(pick_times)))
pick_times = pick_times[iuse_picks]
sta_names = sta_names[iuse_picks]
phase_types = phase_types[iuse_picks]

print('\nUsing picks (%d):\n'%len(pick_times))
print(pick_times)
print(sta_names)
print(phase_types)
print('\n')

xmle, rms_residual, pred_out, arv_p1, arv_s1 = f_apply_location(pick_times, sta_names, phase_types)

print('\n')
print('Reference source (NCEDC): %0.3f lat, %0.3f lon, %0.3f depth, %0.3f s'%(tuple([j for j in ref_src[0,:]])))
print('Located source: %0.3f lat, %0.3f lon, %0.3f depth, %0.3f s'%(tuple([j for j in xmle[0,:]])))


In [ ]:
##### Example 3: #######

# Match two catalogs with space and time constraints
# (Note: catalogs are formated as rows of (n_srcs x 4): (lat, lon, depth, origin time)); where depth is in meters and negative below sea level, origin time in seconds


temporal_match = 10.0 ## seconds
spatial_match = 45e3 ## meters

# Download event from obspy (use europe events)
start_time = UTCDateTime(2008, 1, 15)
end_time = UTCDateTime(2008, 1, 18)
cat1, _, _ = download_catalog([30, 75],[-25, 60], 1.0, start_time, end_time, client = 'USGS', t0 = start_time)
cat2, _, _ = download_catalog([30, 75],[-25, 60], 1.0, start_time, end_time, client = 'GFZ', t0 = start_time)
print('\nUSGS catalog: %d events (M%0.3f min, M%0.3f max)'%(len(cat1), cat1[:,4].min(), cat1[:,4].max()))
print('GFZ catalog: %d events (M%0.3f min, M%0.3f max)\n'%(len(cat2), cat2[:,4].min(), cat2[:,4].max()))

matches = maximize_bipartite_assignment(cat1, cat2, ftrns1, ftrns2, temporal_win = temporal_match, spatial_win = spatial_match)[0]
res = cat1[matches[:,0],0:4] - cat2[matches[:,1],0:4]
## Matched indices stored in matches (matches[:,0] indices of first catalog, matches[:,1] indices of second catalog)
print('\nMatched %d events between %d and %d size catalogs\n'%(len(matches), len(cat1), len(cat2)))
print('Mean residual: %0.3f lat, %0.3f lon, %0.3f depth, %0.3f s'%(res[:,0].mean(), res[:,1].mean(), res[:,2].mean(), res[:,3].mean()))
print('Std residual: %0.3f lat, %0.3f lon, %0.3f depth, %0.3f s'%(res[:,0].std(), res[:,1].std(), res[:,2].std(), res[:,3].std()))

## Compute absolute spatial offset distances using projection function (ftrns1) and the matching indices ()
absolute_offset_cartesian = np.linalg.norm(ftrns1(cat1[matches[:,0],0:3]) - ftrns1(cat2[matches[:,1],0:3]), axis = 1)
print('Average spatial offset: %0.4f (+/- %0.4f) m'%(absolute_offset_cartesian.mean(), absolute_offset_cartesian.std()))
